In [1]:
import threading
import time
from datetime import datetime, timedelta

In [2]:
import threading
import time
from datetime import datetime, timedelta

class TimeSystem:
    def __init__(self):
        self.virtual_time = None  # 延迟设置
        self.real_start_time = None  # 延迟设置
        self.speed = 1.0
        self.running = False  # 初始不运行
        self.lock = threading.Lock()
        self.alarm_callbacks = []

        self.thread = None  # 延迟创建线程

    def _get_virtual_now(self):
        with self.lock:
            if not self.running:
                return self.virtual_time
            elapsed_real = time.time() - self.real_start_time
            virtual_elapsed = elapsed_real * self.speed
            return self.virtual_time + timedelta(seconds=virtual_elapsed)

    def _time_loop(self):
        while True:
            with self.lock:
                if not self.running:
                    break
            time.sleep(0.1)
            now = self._get_virtual_now()
            for callback in self.alarm_callbacks:
                try:
                    callback(now)
                except Exception as e:
                    print(f"回调执行失败: {e}")

    def register_alarm_callback(self, callback):
        """注册一个全局回调，用于接收时间更新并检查闹钟"""
        self.alarm_callbacks.append(callback)

    def set_speed(self, speed):
        with self.lock:
            self.speed = speed
            if self.running:
                self.real_start_time = time.time()

    def pause_time(self):
        with self.lock:
            self.running = False

    def resume_time(self):
        with self.lock:
            if not self.running:
                self.running = True
                self.real_start_time = time.time() - (self._get_virtual_now() - self.virtual_time).total_seconds() / self.speed

    def restart_time(self, year, month, day):
        """重启时间系统，设置新的初始时间"""
        with self.lock:
            self.virtual_time = datetime(year, month, day)
            if self.running:
                self.real_start_time = time.time()

    def start_time(self, year, month, day):
        """显式启动时间系统，并设置初始虚拟时间"""
        with self.lock:
            if self.running:
                print("时间系统已经在运行。")
                return

            # 设置初始虚拟时间
            self.virtual_time = datetime(year, month, day)
            self.real_start_time = time.time()
            self.running = True

            # 启动时间循环线程
            self.thread = threading.Thread(target=self._time_loop, daemon=True)
            self.thread.start()

In [3]:
class UserAlarmManager:
    def __init__(self, time_system):
        self.time_system = time_system
        self.user_alarms = {}  # {user_id: {alarm_id: alarm_data}}
        self.user_counter = 0
        self.lock = threading.Lock()
        time_system.register_alarm_callback(self.check_alarms)

    def create_user(self):
        with self.lock:
            user_id = self.user_counter
            self.user_alarms[user_id] = {}
            self.user_counter += 1
            return user_id

    def add_alarm(self, user_id, hour, minute, repeat=False):
        with self.lock:
            if user_id not in self.user_alarms:
                return None
            alarm_id = len(self.user_alarms[user_id])
            self.user_alarms[user_id][alarm_id] = {
                'time': (hour, minute),
                'callbacks': [],
                'repeat': repeat
            }
            return alarm_id

    def add_alarm_in(self, user_id, days=0, hours=0, minutes=0):
        """
        设置一个只响一次的闹钟，在指定天/小时/分钟后触发
        Args：
            days: 延迟多少天
            hours: 延迟多少小时
            minutes: 延迟多少分钟
        Return：
            alarm_id: 闹钟 ID，或 None 表示失败
        """
        from datetime import timedelta

        current_time = self.time_system._get_virtual_now()
        target_time = current_time + timedelta(days=days, hours=hours, minutes=minutes)
        hour, minute = target_time.hour, target_time.minute

        return self.add_alarm(user_id, hour, minute, repeat=False)

    def remove_alarm(self, user_id, alarm_id):
        with self.lock:
            if user_id not in self.user_alarms or alarm_id not in self.user_alarms[user_id]:
                return False
            del self.user_alarms[user_id][alarm_id]
            return True

    def list_alarms(self, user_id):
        with self.lock:
            if user_id not in self.user_alarms:
                return []
            return [(aid, alarm) for aid, alarm in self.user_alarms[user_id].items()]

    def add_callback_to_alarm(self, user_id, alarm_id, callback):
        with self.lock:
            if user_id not in self.user_alarms or alarm_id not in self.user_alarms[user_id]:
                return False
            self.user_alarms[user_id][alarm_id]['callbacks'].append(callback)
            return True

    def check_alarms(self, current_time):
        with self.lock:
            for user_id, alarms in self.user_alarms.items():
                for alarm_id, alarm in list(alarms.items()):
                    hour, minute = alarm['time']
                    if current_time.hour == hour and current_time.minute == minute and current_time.second == 0:
                        print(f"[{current_time.strftime('%Y-%m-%d %H:%M:%S')}] USER[{user_id}] ALARM ID:{alarm_id} 触发")
                        for cb in alarm['callbacks']:
                            try:
                                cb(user_id, alarm_id, current_time)
                            except Exception as e:
                                print(f"回调执行失败：{e}")
                        if not alarm['repeat']:
                            del self.user_alarms[user_id][alarm_id]

In [ ]:
if __name__ == "__main__":
    ts = TimeSystem()
    manager = UserAlarmManager(ts)

    users = {}
    current_user_id = None  # 当前登录用户

    # 示例回调函数
    def sample_callback(user_id, alarm_id, current_time):
        print(f"🔔 回调函数被触发！用户 {user_id} 的闹钟 {alarm_id} 时间到了：{current_time}")

    def sound_alert(*args):
        print("🔊 嘀嘀嘀...闹钟响了！")

    while True:
        print("\n=== 多用户时间系统 ===")
        print("1. 创建新用户")
        print("2. 切换用户")
        print("3. 控制时间流速")
        print("4. 暂停时间")
        print("5. 恢复时间")
        print("6. 重启时间")
        print("7. 启动时间系统")
        print("8. 添加一次性闹钟（指定多久后）")
        print("9. 添加每日闹钟")
        print("10. 查看当前用户所有闹钟")
        print("11. 为闹钟添加回调函数")  # 👈 新增菜单项
        print("12. 退出")
        choice = input("请选择操作：").strip()

        if choice == "1":
            user_id = manager.create_user()
            users[user_id] = user_id
            print(f"新用户已创建，ID: {user_id}")

        elif choice == "2":
            try:
                user_id = int(input("请输入用户 ID："))
                if user_id not in users:
                    print("用户不存在。")
                    continue
                current_user_id = user_id
                print(f"已切换到用户 {user_id}")
            except ValueError:
                print("请输入有效的用户 ID。")
                
        elif choice == "7":
            print("请输入起始日期：")
            try:
                year = int(input("年份："))
                month = int(input("月份："))
                day = int(input("日期："))
                ts.start_time(year, month, day)
                print(f"时间系统已启动，初始时间为 {year}-{month}-{day}")
            except ValueError:
                print("输入错误，请输入有效的数字。")

        elif choice == "8":
            if current_user_id is None:
                print("请先切换用户。")
                continue
            try:
                days = int(input("延迟天数：") or "0")
                hours = int(input("延迟小时：") or "0")
                minutes = int(input("延迟分钟：") or "0")
                aid = manager.add_alarm_in(current_user_id, days=days, hours=hours, minutes=minutes)
                print(f"一次性闹钟已添加，ID: {aid}")
            except ValueError:
                print("请输入有效的时间参数。")

        elif choice == "9":
            if current_user_id is None:
                print("请先切换用户。")
                continue
            try:
                hour = int(input("小时："))
                minute = int(input("分钟："))
                repeat = input("是否重复？(y/n): ").strip().lower() == 'y'
                aid = manager.add_alarm(current_user_id, hour, minute, repeat)
                print(f"闹钟已添加，ID: {aid}")
            except ValueError:
                print("请输入有效的时间参数。")

        elif choice == "10":
            if current_user_id is None:
                print("请先切换用户。")
                continue
            alarms = manager.list_alarms(current_user_id)
            if not alarms:
                print("该用户没有设置任何闹钟。")
            else:
                print(f"\n--- 用户 {current_user_id} 的闹钟列表 ---")
                for aid, alarm in alarms:
                    time_str = f"{alarm['time'][0]:02}:{alarm['time'][1]:02}"
                    repeat_str = "是" if alarm['repeat'] else "否"
                    cb_count = len(alarm['callbacks'])
                    print(f"ID: {aid} | 时间: {time_str} | 重复: {repeat_str} | 回调数: {cb_count}")
                print("----------------------------------------")

        elif choice == "3":
            try:
                speed = float(input("请输入倍速："))
                ts.set_speed(speed)
                print(f"速度已设为 {speed} 倍速。")
            except ValueError:
                print("请输入有效数字。")

        elif choice == "4":
            ts.pause_time()
            print("时间已暂停。")

        elif choice == "5":
            ts.resume_time()
            print("时间已恢复。")

        elif choice == "6":
            print("请输入新的起始日期以重启时间：")
            try:
                year = int(input("年份："))
                month = int(input("月份："))
                day = int(input("日期："))
                ts.restart_time(year, month, day)
                print(f"时间已重置为 {year}-{month}-{day}")
            except ValueError:
                print("输入错误，请输入有效的数字。")
        elif choice == "11":  # 新增：为闹钟添加回调函数
            if current_user_id is None:
                print("请先切换用户。")
                continue
            
            alarms = manager.list_alarms(current_user_id)
            if not alarms:
                print("该用户没有设置任何闹钟。")
                continue
            
            print("\n--- 用户的闹钟列表 ---")
            for aid, alarm in alarms:
                time_str = f"{alarm['time'][0]:02}:{alarm['time'][1]:02}"
                repeat_str = "是" if alarm['repeat'] else "否"
                cb_count = len(alarm['callbacks'])
                print(f"ID: {aid} | 时间: {time_str} | 重复: {repeat_str} | 回调数: {cb_count}")
            
            try:
                alarm_id = int(input("请输入要添加回调的闹钟 ID："))
                print("选择要添加的回调函数：")
                print("1. 默认提示")
                print("2. 声音提醒")
                cb_choice = input("请输入编号：").strip()
                
                if cb_choice == "1":
                    result = manager.add_callback_to_alarm(current_user_id, alarm_id, sample_callback)
                elif cb_choice == "2":
                    result = manager.add_callback_to_alarm(current_user_id, alarm_id, sound_alert)
                else:
                    print("无效选项。")
                    continue
                
                if result:
                    print(f"回调函数已成功添加到闹钟 {alarm_id}。")
                else:
                    print(f"无法找到闹钟 ID {alarm_id} 或已存在此回调。")
                    
            except ValueError:
                print("请输入有效数字。")
                
        elif choice == "12":
            print("退出程序...")
            break

        else:
            print("无效选项。")

        # 其他菜单项保持不变...


=== 多用户时间系统 ===
1. 创建新用户
2. 切换用户
3. 控制时间流速
4. 暂停时间
5. 恢复时间
6. 重启时间
7. 启动时间系统
8. 添加一次性闹钟（指定多久后）
9. 添加每日闹钟
10. 查看当前用户所有闹钟
11. 为闹钟添加回调函数
12. 退出


请选择操作： 1


新用户已创建，ID: 0

=== 多用户时间系统 ===
1. 创建新用户
2. 切换用户
3. 控制时间流速
4. 暂停时间
5. 恢复时间
6. 重启时间
7. 启动时间系统
8. 添加一次性闹钟（指定多久后）
9. 添加每日闹钟
10. 查看当前用户所有闹钟
11. 为闹钟添加回调函数
12. 退出


请选择操作： 1


新用户已创建，ID: 1

=== 多用户时间系统 ===
1. 创建新用户
2. 切换用户
3. 控制时间流速
4. 暂停时间
5. 恢复时间
6. 重启时间
7. 启动时间系统
8. 添加一次性闹钟（指定多久后）
9. 添加每日闹钟
10. 查看当前用户所有闹钟
11. 为闹钟添加回调函数
12. 退出


...

回调函数已成功添加到闹钟 0。

=== 多用户时间系统 ===
1. 创建新用户
2. 切换用户
3. 控制时间流速
4. 暂停时间
5. 恢复时间
6. 重启时间
7. 启动时间系统
8. 添加一次性闹钟（指定多久后）
9. 添加每日闹钟
10. 查看当前用户所有闹钟
11. 为闹钟添加回调函数
12. 退出


请选择操作： 7


请输入起始日期：


年份： 2008
月份： 1
日期： 1


时间系统已启动，初始时间为 2008-1-1

=== 多用户时间系统 ===
1. 创建新用户
2. 切换用户
3. 控制时间流速
4. 暂停时间
5. 恢复时间
6. 重启时间
7. 启动时间系统
8. 添加一次性闹钟（指定多久后）
9. 添加每日闹钟
10. 查看当前用户所有闹钟
11. 为闹钟添加回调函数
12. 退出
